# Exercise 49 — CSV with the `csv` module

## Concept

The stdlib `csv` module handles the messy reality of CSVs: quoting, embedded commas, embedded newlines, escaping. Don't `.split(",")` — it breaks on the first row with a comma inside a quoted field.

```python
import csv

# Read rows as lists
with open("data.csv", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)            # first row
    for row in reader:
        print(row)

# Read rows as dicts (most useful)
with open("data.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["name"], row["email"])

# Write
with open("out.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "name"])
    writer.writeheader()
    writer.writerows([{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}])
```

**`newline=""`** — critical on Windows. Without it, you'll get blank lines between rows on write, and incorrect splitting on read when fields contain embedded newlines.

**Quoting/delimiters**: `csv.reader(f, delimiter=";", quotechar='"')` for European-style CSVs.

All values come back as **strings** — `DictReader` doesn't convert types. You parse `int(row["age"])` yourself, or use pandas (later).

## Setup

Creates `users.csv` with a tricky row (embedded comma + quoted name) so you really see why `.split(",")` is wrong.

In [ ]:
from pathlib import Path

CSV = Path("users.csv")
CSV.write_text(
    'id,name,city\n'
    '1,Alice,Curitiba\n'
    '2,"Souza, Bob",Rio\n'
    '3,Carol,Sao Paulo\n'
    '4,Diana,"Rio de Janeiro"\n',
    encoding="utf-8",
)
print("wrote", CSV.resolve())


## Task 1 — Read with `DictReader`

Write a function that reads a CSV and returns all rows as a list of dicts.

```python
def read_users(path: str) -> list[dict]:
    ...
```

Expected output for `users.csv` (above):
```
[
    {"id": "1", "name": "Alice",       "city": "Curitiba"},
    {"id": "2", "name": "Souza, Bob",  "city": "Rio"},        # comma preserved
    {"id": "3", "name": "Carol",       "city": "Sao Paulo"},
    {"id": "4", "name": "Diana",       "city": "Rio de Janeiro"},
]
```

Note: values are strings. Don't convert `id` to `int` here — that's a separate concern.

In [ ]:
import csv

def read_users(path: str) -> list[dict]:
    pass  # TODO

rows = read_users("users.csv")
assert rows == [
    {"id": "1", "name": "Alice",      "city": "Curitiba"},
    {"id": "2", "name": "Souza, Bob", "city": "Rio"},
    {"id": "3", "name": "Carol",      "city": "Sao Paulo"},
    {"id": "4", "name": "Diana",      "city": "Rio de Janeiro"},
]
print("ok")


## Task 2 — Stream rows as a generator

Return a **generator** of rows (dicts) instead of a list. Same data, but lazy — usable on a 10GB CSV without loading it all.

```python
def iter_users(path: str):
    ...
```

Gotcha: if you `yield` inside `with open(...) as f`, the file stays open until the generator is exhausted (or garbage-collected). That's fine — actually what you want — but means callers should iterate the generator promptly.

In [ ]:
import csv

def iter_users(path: str):
    pass  # TODO

gen = iter_users("users.csv")
first = next(gen)
assert first == {"id": "1", "name": "Alice", "city": "Curitiba"}
remaining = list(gen)
assert len(remaining) == 3
print("ok")


## Task 3 — Write a CSV with `DictWriter`

Given a list of records, write them to a CSV with header row. Field order in the output must match the `fieldnames` parameter.

```python
def write_records(path: str, records: list[dict], fieldnames: list[str]) -> None:
    ...
```

What happens if a record has an extra key not in `fieldnames`? By default `DictWriter` raises `ValueError`. Use `extrasaction="ignore"` if you want to silently drop them — but **don't** do that here. Let it raise; the assertion below relies on the strict default.

In [ ]:
import csv
from pathlib import Path

def write_records(path: str, records: list[dict], fieldnames: list[str]) -> None:
    pass  # TODO

write_records(
    "out_simple.csv",
    [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Souza, Bob"}],
    fieldnames=["id", "name"],
)

text = Path("out_simple.csv").read_text(encoding="utf-8")
assert text == 'id,name\n1,Alice\n2,"Souza, Bob"\n'
print("ok")


## Task 4 — Custom delimiter

Given a TSV file (tab-separated), read it with `csv.reader` using `delimiter="\t"`. Return a list of `(name, age)` tuples with `age` parsed to `int`.

```python
def read_people_tsv(path: str) -> list[tuple[str, int]]:
    ...
```

Assume the file has a header row.

Edge case: if `age` can't be parsed, skip that row (don't raise). This is a small DE-style "be permissive on read" pattern.

In [ ]:
import csv
from pathlib import Path

Path("people.tsv").write_text(
    "name\tage\n"
    "Alice\t30\n"
    "Bob\tnot_a_number\n"
    "Carol\t25\n",
    encoding="utf-8",
)

def read_people_tsv(path: str) -> list[tuple[str, int]]:
    pass  # TODO

assert read_people_tsv("people.tsv") == [("Alice", 30), ("Carol", 25)]
print("ok")
